# tSCS EMG — the four comparison figures (subject NTA, 24-07-2026)

One session, one participant: **stimulation mode** (30 Hz burst / ARC-EX) × **polarity**
(cathodic / anodic) × **lidocaine** (before / with). Four figures, each holding one thing fixed:

| figure | fixed | compared | with |
|---|---|---|---|
| **1** | anodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **2** | cathodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **3** | ARC-EX | anodic vs cathodic | before and with lidocaine |
| **4** | 30 Hz burst | anodic vs cathodic | before and with lidocaine |

## Colours and styles
**gray = before lidocaine, orange = with lidocaine** in every figure. The second factor is the
**style**: in figures 1–2 **solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX**; in
figures 3–4 **solid / plain = cathodic, dashed / hatched = anodic**.

**Each figure has its own settings** in `FIGS` (config cell): the **intensity per compared value**
and the **muscle** for its one-muscle version. Defaults: burst 35 mA, ARC-EX 110 mA — different mA,
roughly matched relative to their motor thresholds (burst 25–30, ARC-EX 65–90 mA from the log);
figures 3 and 4 compare one mode with itself, so both their values sit at the same intensity.

Every figure also comes with **waterfalls** — all intensities of each condition stacked, and the
four overlaid — so you can see the whole sweep before choosing the mA to compare at.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty, waterfall, waterfall_overlay
from functions.burst import compare_at_intensity, summary_curves, plot_pulse_overlay
from functions.paper import fig_train_modes
set_style()


## Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
FILE = {   # (mode, polarity, lidocaine state) -> file
    ("burst", "cathodic", "before"):    "Burst_autosave_20260724_102443_670ms.csv",
    ("burst", "cathodic", "lidocaine"): "Burst_autosave_20260724_113834_522ms.csv",
    ("burst", "anodic",   "before"):    "Burst_autosave_20260724_102721_694ms.csv",
    ("burst", "anodic",   "lidocaine"): "Burst_autosave_20260724_114055_891ms.csv",
    ("arcex", "cathodic", "before"):    "Modulated_autosave_20260724_103530_508ms.csv",
    ("arcex", "cathodic", "lidocaine"): "Modulated_autosave_20260724_114332_473ms.csv",
    ("arcex", "anodic",   "before"):    "Modulated_autosave_20260724_103849_459ms.csv",
    ("arcex", "anodic",   "lidocaine"): "Modulated_autosave_20260724_114730_561ms.csv",
}
NAME = {"burst": "30 Hz", "arcex": "ARC-EX", "cathodic": "cathodic", "anodic": "anodic"}
LIDO_COL = {"before": "0.45", "lidocaine": "#f39c12"}       # colour = lidocaine
STYLE_A  = ("",    "-")                                      # first compared value: plain / solid
STYLE_B  = ("///", "--")                                     # second: hatched / dashed

FIGS = {   # <-- one entry per figure: intensity of each compared value, and the muscle to zoom on
    1: dict(fixed=("polarity", "anodic"),   compare=("burst", "arcex"),
            amps={"burst": 35, "arcex": 110}, muscle="Flex. digitorum (R)"),
    2: dict(fixed=("polarity", "cathodic"), compare=("burst", "arcex"),
            amps={"burst": 35, "arcex": 110}, muscle="Flex. digitorum (R)"),
    3: dict(fixed=("mode", "arcex"),        compare=("cathodic", "anodic"),
            amps={"cathodic": 110, "anodic": 110}, muscle="Flex. digitorum (R)"),
    4: dict(fixed=("mode", "burst"),        compare=("cathodic", "anodic"),
            amps={"cathodic": 35, "anodic": 35}, muscle="Flex. digitorum (R)"),
}
XLIM_WF   = (-20, 130)      # time window of the waterfalls (ms)
WF_EACH   = True            # also draw each condition's waterfall on its own, not only the overlay
WF_MUSCLES = None           # None = all muscles in the waterfalls, or a list, e.g. ["Flex. digitorum (R)"]

N_PULSES, RESP_START_MS, GUARD_MS, MIN_SNR, MAX_EDGE_FRAC = 10, 8.0, 1.0, 2.0, 0.5
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC)

muscles = [c for c in load_run(D + FILE[("burst", "cathodic", "before")])[2] if c != "Trigger A"]

def build(n):
    """Everything figure `n` needs, ordered A-before, A-lidocaine, B-before, B-lidocaine."""
    spec = FIGS[n]; fixed, compare = spec["fixed"], spec["compare"]
    files, labels, colours, hatches, lss, amps = [], [], [], [], [], []
    for val, (hat, ls) in zip(compare, (STYLE_A, STYLE_B)):
        for state in ("before", "lidocaine"):
            key = (val, fixed[1], state) if fixed[0] == "polarity" else (fixed[1], val, state)
            files.append(D + FILE[key])
            labels.append(f"{NAME[val]} · {state}")
            colours.append(LIDO_COL[state]); hatches.append(hat); lss.append(ls)
            amps.append(spec["amps"][val])
    return dict(files=files, labels=labels, colours=colours, hatches=hatches, linestyles=lss,
                amps=tuple(amps), muscle=spec["muscle"], n=n)

def show(cfg, title, muscle=None):
    """The comparison figure; muscle=cfg["muscle"] for the one-muscle version."""
    compare_at_intensity(cfg["files"], None, amp=cfg["amps"], normalize="none", muscles=muscle,
                         labels=cfg["labels"], colours=cfg["colours"], hatches=cfg["hatches"],
                         linestyles=cfg["linestyles"], title=title, **KW)

def wf(cfg, title, muscles_=None, each=None):
    """Waterfalls of the four conditions: each on its own (optional) and all overlaid, one shared
    gain per muscle so the amplitudes are comparable."""
    ms = muscles_ or WF_MUSCLES or muscles
    runs = [load_run(f) for f in cfg["files"]]
    print(f"===== {title} - {cfg['labels'][0]} (sets the gain)")
    gains = waterfall(*runs[0], ms, xlim=XLIM_WF)
    if WF_EACH if each is None else each:
        for lab, r in zip(cfg["labels"][1:], runs[1:]):
            print(f"===== {title} - {lab}")
            waterfall(*r, ms, xlim=XLIM_WF, gains=gains)
    print(f"===== {title} - overlay")
    waterfall_overlay(runs, muscles=ms, xlim=XLIM_WF, gains=gains, labels=cfg["labels"],
                      colours=cfg["colours"], linestyles=cfg["linestyles"])

for n, spec in FIGS.items():                      # check every requested intensity exists
    for val in spec["compare"]:
        for state in ("before", "lidocaine"):
            key = (val, spec["fixed"][1], state) if spec["fixed"][0] == "polarity" else (spec["fixed"][1], val, state)
            meta, _, _ = load_run(D + FILE[key])
            amps = [m["amp_ma"] for m in meta]; a = spec["amps"][val]
            print(f"fig {n} | {NAME[key[0]]:7s} {key[1]:9s} {key[2]:10s} {a:4d} mA "
                  + ("ok" if a in amps else f"MISSING - available: {amps}"))


## Figure 1 · 30 Hz vs ARC-EX — **anodic**, before and with lidocaine

Solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX; gray = before, orange = with lidocaine. Intensity and muscle for this figure: `FIGS[1]` in the config.

In [ ]:
FIG1 = build(1)
show(FIG1, "Fig 1 · anodic — 30 Hz vs ARC-EX")                       # all muscles


### Figure 1 · one muscle (`FIGS[1]['muscle']`)

In [ ]:
show(FIG1, f"Fig 1 · anodic — 30 Hz vs ARC-EX — {FIG1['muscle']}", muscle=FIG1["muscle"])


### Figure 1 · waterfalls — every intensity of each condition

Set `WF_MUSCLES` (config) to a list to restrict which muscles are drawn, or pass `muscles_=` here.

In [ ]:
wf(FIG1, "Fig 1")


## Figure 2 · 30 Hz vs ARC-EX — **cathodic**, before and with lidocaine

Same styles as figure 1. Intensity and muscle for this figure: `FIGS[2]` in the config.

In [ ]:
FIG2 = build(2)
show(FIG2, "Fig 2 · cathodic — 30 Hz vs ARC-EX")                       # all muscles


### Figure 2 · one muscle (`FIGS[2]['muscle']`)

In [ ]:
show(FIG2, f"Fig 2 · cathodic — 30 Hz vs ARC-EX — {FIG2['muscle']}", muscle=FIG2["muscle"])


### Figure 2 · waterfalls — every intensity of each condition

Set `WF_MUSCLES` (config) to a list to restrict which muscles are drawn, or pass `muscles_=` here.

In [ ]:
wf(FIG2, "Fig 2")


## Figure 3 · ARC-EX — **anodic vs cathodic**, before and with lidocaine

Solid / plain = cathodic, dashed / hatched = anodic. Intensity and muscle for this figure: `FIGS[3]` in the config.

In [ ]:
FIG3 = build(3)
show(FIG3, "Fig 3 · ARC-EX — cathodic vs anodic")                       # all muscles


### Figure 3 · one muscle (`FIGS[3]['muscle']`)

In [ ]:
show(FIG3, f"Fig 3 · ARC-EX — cathodic vs anodic — {FIG3['muscle']}", muscle=FIG3["muscle"])


### Figure 3 · waterfalls — every intensity of each condition

Set `WF_MUSCLES` (config) to a list to restrict which muscles are drawn, or pass `muscles_=` here.

In [ ]:
wf(FIG3, "Fig 3")


### Figure 3 · across intensities

In [ ]:
summary_curves(FIG3["files"], None, labels=FIG3["labels"], colours=FIG3["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Figure 4 · 30 Hz burst — **anodic vs cathodic**, before and with lidocaine

Solid / plain = cathodic, dashed / hatched = anodic. Intensity and muscle for this figure: `FIGS[4]` in the config.

In [ ]:
FIG4 = build(4)
show(FIG4, "Fig 4 · 30 Hz burst — cathodic vs anodic")                       # all muscles


### Figure 4 · one muscle (`FIGS[4]['muscle']`)

In [ ]:
show(FIG4, f"Fig 4 · 30 Hz burst — cathodic vs anodic — {FIG4['muscle']}", muscle=FIG4["muscle"])


### Figure 4 · waterfalls — every intensity of each condition

Set `WF_MUSCLES` (config) to a list to restrict which muscles are drawn, or pass `muscles_=` here.

In [ ]:
wf(FIG4, "Fig 4")


### Figure 4 · across intensities

In [ ]:
summary_curves(FIG4["files"], None, labels=FIG4["labels"], colours=FIG4["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Paper-style version — traces + pulse 1 vs rest, 2–3 muscles

`fig_train_modes` for any pair of conditions; set `SAVE` to write a 300 dpi PNG.

In [ ]:
MUSCLES_FIG = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]
SAVE = None        # e.g. "figures/fig1_anodic_burst_vs_arcex.png"

specs = [dict(label="30 Hz burst", csv=D + FILE[("burst", "anodic", "before")], amp=FIGS[1]["amps"]["burst"], colour="black"),
         dict(label="ARC-EX",      csv=D + FILE[("arcex", "anodic", "before")], amp=FIGS[1]["amps"]["arcex"], colour="#d62728")]
fig_train_modes(specs, MUSCLES_FIG, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS,
                title="anodic · before lidocaine", save=SAVE);
